# F1Compiler 🏎️

**Lenguaje de programación basado en telemetría y estrategias de Fórmula 1.**

Este notebook ejecuta todas las fases del compilador interactuando con los archivos locales del proyecto, preservando su arquitectura modular.

### Preparación del Entorno
Antes de ejecutar las celdas de este Notebook, asegúrate de haber seguido las instrucciones de instalación detalladas en el **`README.md`**.

Esto incluye:
1. Instalar **Graphviz** en tu sistema.
2. Crear y activar tu **entorno virtual** (`.venv`).
3. Instalar las dependencias de Python con `pip install -r requirements.txt`.

### Instrucciones de Ejecución
1. Asegúrate de estar corriendo el kernel de Jupyter utilizando tu entorno `.venv`.
2. Ejecuta las celdas secuencialmente de arriba a abajo.
3. Los archivos fuente evaluados están en `examples/` y los módulos en `src/tools/`.


## Fase 1: Análisis Léxico
Ejecuta el lexer sobre el programa de ejemplo, mostrando la tabla detallada de tokens reconocidos.

In [1]:
!python ejecutar_lexer.py examples/estrategia_carrera.txt


Programa de ejemplo definido (1607 caracteres).
ANÁLISIS LÉXICO — Lista completa de tokens
Num   Tipo                      Texto                          Línea    Columna
--------------------------------------------------------------------------------
1     GREEN_LIGHT               'green_light'                  2        0
3     TYPE_LAPS                 'laps'                         5        4
39    ID                        'current_lap'                  5        9
29    ASSIGN                    '='                            5        21
37    NUM_INT                   '1'                            5        23
30    SEMI                      ';'                            5        24
3     TYPE_LAPS                 'laps'                         6        4
39    ID                        'total_laps'                   6        9
29    ASSIGN                    '='                            6        20
37    NUM_INT                   '5'                            6        22
30 

## Fase 2: Análisis Sintáctico (AST)
Genera el Árbol de Sintaxis Abstracta (AST) usando Graphviz para visualizar las relaciones del código fuente.

In [2]:
!python src/tools/herramientas_ast.py examples/estrategia_carrera.txt


Generando telemetría visual del AST...
Árbol en formato texto:
(program green_light (statement (declaracion (tipo telemetry) expected_pace = (expresion (expresion 85.5) throttle (expresion (expresion 2.0) catch_slipstream (expresion 1.5))) ;)) (statement (declaracion (tipo laps) current_lap = (expresion 1) ;)) (statement (ciclo start_new_stint ( (asignacion current_lap = (expresion 1) ;) up_to_lap (expresion 5) ) { (statement (impresion engineer_says ( (expresion "Pushing on new stint") ) ;)) })) chequered_flag <EOF>)

Imagen generada y abierta como 'estrategia_f1_ast.png'


## Fase 3: Análisis Semántico
Evalúa las reglas semánticas (Dirección de Carrera) detectando variables duplicadas, no declaradas y sin uso.

In [ ]:
!python src/tools/herramientas_semanticas.py examples/estrategia_carrera.txt


REPORTE DE DIRECCIÓN DE CARRERA (Análisis Semántico)
Grid Penalty (Línea 5): El parámetro 'current_lap' ya estaba registrado.
Black Flag (Línea 8): Intento de reasignar 'fuel_load', pero no existe en telemetría.
Black Flag (Línea 9): Se intentó leer 'unknown_tyre', pero no está registrado.
Warning (Línea 4): El parámetro 'current_lap' fue declarado pero nunca se utilizó en la estrategia.
Warning (Línea 12): El parámetro 'neumatico' fue declarado pero nunca se utilizó en la estrategia.

Tabla de Símbolos Final:
  - current_lap: {'tipo': 'laps', 'linea': 4, 'usado': False}
  - neumatico: {'tipo': 'compound', 'linea': 12, 'usado': False}
  - pace: {'tipo': 'telemetry', 'linea': 14, 'usado': True}


## Fase 4: Generación de Código (Transpilación) y Ejecución
Traduce el código fuente F1Compiler a Python puro y lo ejecuta dinámicamente con exec().

## Fase 4.1: Ejecución Alternativa (Escenario de Lluvia)
Probamos el compilador con la estrategia alternativa `estrategia_lluvia.txt` para validar el nuevo ciclo WHILE y los condicionales dinámicos.

In [5]:
!python generador_codigo.py examples/estrategia_lluvia.txt

Leyendo programa desde: examples/estrategia_lluvia.txt
CÓDIGO TRADUCIDO (PYTHON):
# ==========================================
# CÓDIGO GENERADO AUTOMÁTICAMENTE (F1Compiler)
# ==========================================
import math

# --- Funciones Nativas de Telemetría ---
def tyre_wear_calc(x): return math.sqrt(x)
def calc_steering(x): return math.cos(math.radians(x))
def deploy_ers(x, y): return math.pow(x, y)
def calculate_undercut_delta(x): return math.log(x)

# --- Estrategia Principal ---
delta_time = 0.0
start_tyre = "Medium"
rain_expected = True
print("--- Starting Wet Race Simulation ---")
print("Starting on:")
print(start_tyre)
for current_lap in range(int(1), int(4) + 1):
    if current_lap == 3:
        print("Radar shows rain approaching!")
        rain_expected = False
    else:
        print("Track is still dry, keep pushing.")
        delta_time = delta_time + 0.2
if rain_expected == False:
    print("Box, box! Changing to Wet tyres.")
    new_tyre = "Wet"
    delta_tim

In [6]:
!python generador_codigo.py examples/estrategia_carrera.txt


Leyendo programa desde: examples/estrategia_carrera.txt
CÓDIGO TRADUCIDO (PYTHON):
# ==========================================
# CÓDIGO GENERADO AUTOMÁTICAMENTE (F1Compiler)
# ==========================================
import math

# --- Funciones Nativas de Telemetría ---
def tyre_wear_calc(x): return math.sqrt(x)
def calc_steering(x): return math.cos(math.radians(x))
def deploy_ers(x, y): return math.pow(x, y)
def calculate_undercut_delta(x): return math.log(x)

# --- Estrategia Principal ---
base_pace = 85.5
current_tyre = "Soft"
print("--- Starting Race Simulation ---")
for current_lap in range(int(1), int(3) + 1):
    print("Lap number:")
    print(current_lap)
    if current_lap == 2 and True:
        print("Pushing! DRS Enabled")
        base_pace = base_pace * 1.05
    else:
        base_pace = base_pace + 0.1
final_wear = tyre_wear_calc(base_pace)
print("Final Telemetry:")
print(final_wear)
is_raining = True
while not is_raining:
    print("Track is dry. Keeping the pace.")
 

## Casos de Prueba Adicionales
Ejecuta diferentes escenarios de código de ejemplo para validar los distintos componentes lógicos y cíclicos de la gramática.

In [4]:
!python casos_prueba.py


  Prueba 1 — Tipos de Datos y Setup
Num   Tipo                      Texto                          Línea    Columna
--------------------------------------------------------------------------------
1     GREEN_LIGHT               'green_light'                  2        0
6     TYPE_COMPOUND             'compound'                     4        4
39    ID                        'current_tyre'                 4        13
29    ASSIGN                    '='                            4        26
38    STRING                    '"Hard"'                       4        28
30    SEMI                      ';'                            4        34
4     TYPE_TELEMETRY            'telemetry'                    5        4
39    ID                        'fuel_load'                    5        14
29    ASSIGN                    '='                            5        24
36    NUM_FLOAT                 '105.5'                        5        26
30    SEMI                      ';'                    